In [11]:
import tomllib
from pathlib import Path

import pandas as pd
import snowflake.connector

import requests

from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_airports_in_Sweden"
headers = {"User-Agent": "student-data-engineering-project/1.0"}

html = requests.get(url, headers=headers).text
tables = pd.read_html(StringIO(html))

for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:4])

secrets_path = Path("..") / ".dlt" / "secrets.toml"
creds = tomllib.loads(secrets_path.read_text())["destination"]["snowflake"]["credentials"]

conn = snowflake.connector.connect(
    account=creds["host"],
    user=creds["username"],
    password=creds["password"],
    role=creds["role"],
    warehouse=creds["warehouse"],
    database=creds["database"],
    schema="STAGING",
)

cur = conn.cursor()
cur.execute("SELECT * FROM FLIGHTS")
df = pd.DataFrame(cur.fetchall(), columns=[c[0].lower() for c in cur.description])
cur.close()
conn.close()

print(df.shape)
print(df.columns.tolist())

0 (169, 7) ['City\xa0served', 'ICAO', 'IATA', 'Airport\xa0name']
1 (28, 5) ['City\xa0served', 'ICAO', 'IATA', 'Airport\xa0name']
2 (3, 2) ['vteLists of airports in Europe', 'vteLists of airports in Europe.1']
(3163, 48)
['flight_id', 'departure_airport_swedish', 'departure_airport_english', 'di_indicator', 'airport', 'flight_date', 'direction', 'flight_leg_identifier__callsign', 'flight_leg_identifier__flight_id', 'flight_leg_identifier__flight_departure_date_utc', 'flight_leg_identifier__departure_airport_iata', 'flight_leg_identifier__arrival_airport_iata', 'flight_leg_identifier__departure_airport_icao', 'flight_leg_identifier__arrival_airport_icao', 'location_and_status__terminal', 'location_and_status__flight_leg_status', 'location_and_status__flight_leg_status_swedish', 'location_and_status__flight_leg_status_english', 'arrival_time__scheduled_utc', 'airline_operator__iata', 'airline_operator__icao', 'airline_operator__name', '_dlt_load_id', '_dlt_id', 'baggage__baggage_claim_uni

In [12]:
df.info()
df.head()


<class 'pandas.DataFrame'>
RangeIndex: 3163 entries, 0 to 3162
Data columns (total 48 columns):
 #   Column                                            Non-Null Count  Dtype                              
---  ------                                            --------------  -----                              
 0   flight_id                                         3163 non-null   str                                
 1   departure_airport_swedish                         1583 non-null   str                                
 2   departure_airport_english                         1583 non-null   str                                
 3   di_indicator                                      3163 non-null   str                                
 4   airport                                           3163 non-null   str                                
 5   flight_date                                       3163 non-null   str                                
 6   direction                                  

,flight_id,departure_airport_swedish,departure_airport_english,di_indicator,airport,flight_date,direction,flight_leg_identifier__callsign,flight_leg_identifier__flight_id,flight_leg_identifier__flight_departure_date_utc,...,check_in__check_in_status_swedish,check_in__check_in_status_english,location_and_status__gate_action,location_and_status__gate_action_swedish,location_and_status__gate_action_english,location_and_status__gate_open_utc,location_and_status__gate_close_utc,departure_time__actual_utc,departure_time__estimated_utc,baggage__estimated_first_bag_utc
0,TO7440,Paris ORY,Paris ORY,S,ARN,2026-10-09,arrivals,TVF34AJ,TO7440,2026-10-08 17:00:00-07:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,NaT
1,FR4464,Birmingham,Birmingham,I,ARN,2026-10-09,arrivals,RYR4464,FR4464,2026-10-08 17:00:00-07:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,NaT
2,FR317,Porto,Porto,S,ARN,2026-10-09,arrivals,RYR7889,FR317,2026-10-08 17:00:00-07:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,NaT
3,FR3008,Dublin,Dublin,I,ARN,2026-10-09,arrivals,RYR3008,FR3008,2026-10-08 17:00:00-07:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,NaT
4,FR7686,Brindisi,Brindisi,S,ARN,2026-10-09,arrivals,RYR62UM,FR7686,2026-10-08 17:00:00-07:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,NaT


In [13]:
missing = df.isna().mean().sort_values(ascending=False) * 100
missing.round(1)

departure_time__estimated_utc                       97.0
baggage__estimated_first_bag_utc                    94.3
location_and_status__gate_open_utc                  71.0
baggage__last_bag_utc                               70.8
baggage__first_bag_utc                              70.5
arrival_time__actual_utc                            67.9
location_and_status__gate_action_english            67.6
location_and_status__gate_action_swedish            67.6
departure_time__actual_utc                          67.2
location_and_status__gate_close_utc                 67.1
check_in__check_in_status_english                   67.0
check_in__check_in_status_swedish                   67.0
check_in__check_in_status                           67.0
location_and_status__gate_action                    66.7
arrival_time__estimated_utc                         65.0
check_in__check_in_desk_from                        60.5
check_in__check_in_desk_to                          60.5
baggage__baggage_claim_unit    

In [14]:
df.duplicated(subset=["flight_id", "airport", "flight_date", "direction"]).sum()


np.int64(0)

In [15]:
df.groupby(["airport", "direction"]).size().unstack()


direction,arrivals,departures
airport,,
ARN,1052,1048
BMA,66,66
GOT,231,227
KRN,6,6
LLA,46,46
MMX,67,68
OSD,17,19
RNB,16,17
UME,44,45


In [16]:
df["airline_operator__name"].value_counts().head(10)

airline_operator__name
SAS Scandinavian Airlines    1193
Norwegian                     407
Ryanair Ltd                   293
Finnair                       113
Lufthansa                      93
FROST Air                      87
Nyxair                         84
KLM                            66
Wizz Air                       62
PopulAir                       62
Name: count, dtype: int64

In [17]:
df["location_and_status__flight_leg_status"].value_counts()


location_and_status__flight_leg_status
ACT    1034
LAN    1014
DEL     539
SCH     489
SEQ      60
CAN      27
Name: count, dtype: int64

In [18]:
[c for c in df.columns if "time" in c]

['arrival_time__scheduled_utc',
 'arrival_time__estimated_utc',
 'arrival_time__actual_utc',
 'departure_time__scheduled_utc',
 'departure_time__actual_utc',
 'departure_time__estimated_utc']

In [19]:
def pick(arrival_col, departure_col):
    a = df[arrival_col] if arrival_col in df.columns else pd.Series(pd.NaT, index=df.index)
    d = df[departure_col] if departure_col in df.columns else pd.Series(pd.NaT, index=df.index)
    return pd.to_datetime(a.combine_first(d), utc=True)


scheduled = pick("arrival_time__scheduled_utc", "departure_time__scheduled_utc")
estimated = pick("arrival_time__estimated_utc", "departure_time__estimated_utc")
actual = pick("arrival_time__actual_utc", "departure_time__actual_utc")

print("Planerad tid:", scheduled.notna().sum())
print("Beräknad tid:", estimated.notna().sum())
print("Faktisk tid:", actual.notna().sum())
print("Totalt antal flyg:", len(df))

Planerad tid: 3163
Beräknad tid: 1203
Faktisk tid: 2050
Totalt antal flyg: 3163


In [20]:
best_time = actual.combine_first(estimated)
df["delay_min"] = (best_time - scheduled).dt.total_seconds() / 60

df["delay_min"].describe()


count    2151.000000
mean        0.619665
std        23.213349
min       -46.050000
25%        -9.466667
50%        -3.000000
75%         4.775000
max       404.066667
Name: delay_min, dtype: float64

In [21]:

url = "https://en.wikipedia.org/wiki/List_of_airports_in_Sweden"
headers = {"User-Agent": "student-data-engineering-project/1.0"}

html = requests.get(url, headers=headers).text
tables = pd.read_html(StringIO(html))

for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:4])

0 (169, 7) ['City\xa0served', 'ICAO', 'IATA', 'Airport\xa0name']
1 (28, 5) ['City\xa0served', 'ICAO', 'IATA', 'Airport\xa0name']
2 (3, 2) ['vteLists of airports in Europe', 'vteLists of airports in Europe.1']


In [22]:
print(tables[0].head(8))
print(tables[1].head(5))


  City served  ICAO IATA              Airport name     Usage  \
0    Alingsås  ESGI  NaN          Alingsås Airport    Public   
1  Anderstorp  ESMP  NaN        Anderstorp Airport    Public   
2      Arboga  ESQO  NaN            Arboga Airport    Public   
3       Arbrå  ESUB  NaN             Arbrå Airport    Public   
4  Arvidsjaur  ESNX  AJR        Arvidsjaur Airport    Public   
5      Arvika  ESKV  NaN  Arvika–Westlanda Airport    Public   
6      Avesta  ESVA  NaN            Avesta Airport    Public   
7       Berga  ESQP  NaN             Berga Airport  Military   

                Runway(s)  Passengers (2023)[1]  
0     01/19, 800 m, Grass                   NaN  
1  04/22, 1000 m, Asphalt                   NaN  
2  15/33, 2000 m, Asphalt                   NaN  
3     18/36, 700 m, Grass                   NaN  
4  12/30, 2500 m, Asphalt               55355.0  
5  01/19, 1150 m, Asphalt                   NaN  
6     15/33, 850 m, Grass                   NaN  
7            183 m, Pav

In [24]:
airports = tables[0].copy()

airports.columns = (
    airports.columns.str.replace("\xa0", " ")
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
airports = airports.rename(columns={"runway_s": "runways"})

for col in airports.select_dtypes(exclude="number").columns:
    airports[col] = (
        airports[col]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.replace("\xa0", " ")
        .str.strip()
    )
    airports[col] = airports[col].mask(airports[col].isin(["nan", "None", ""]))

airports["passengers_2023"] = pd.to_numeric(airports["passengers_2023"], errors="coerce")

print(airports.columns.tolist())
print(airports.shape)
print("Flygplatser med IATA-kod:", airports["iata"].notna().sum())
airports[airports["iata"].isin(["ARN", "BMA", "GOT", "MMX", "LLA", "UME", "OSD", "VBY", "RNB", "KRN"])][["iata", "airport_name", "city_served"]]

['city_served', 'icao', 'iata', 'airport_name', 'usage', 'runways', 'passengers_2023']
(169, 7)
Flygplatser med IATA-kod: 52


,iata,airport_name,city_served
33,GOT,Göteborg Landvetter Airport,Gothenburg
65,KRN,Kiruna Airport,Kiruna
83,LLA,Luleå/Kallax Airport,Luleå
86,MMX,Malmö Airport,Malmö
103,RNB,Ronneby Airport (F 17),Ronneby
117,ARN,Stockholm–Arlanda Airport,Stockholm
118,BMA,Stockholm–Bromma Airport,Stockholm
142,UME,Umeå Airport,Umeå
148,VBY,Visby Airport,Visby
167,OSD,Åre Östersund Airport (F4 Frösön Airbase),Östersund


In [25]:
import string
import time

airline_frames = []

for letter in string.ascii_uppercase:
    url = f"https://en.wikipedia.org/wiki/List_of_airline_codes_({letter})"
    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        print(letter, "status", r.status_code)
        continue

    for t in pd.read_html(StringIO(r.text)):
        if "IATA" in [str(c) for c in t.columns]:
            airline_frames.append(t)

    time.sleep(0.5)

airlines = pd.concat(airline_frames, ignore_index=True)
print(airlines.shape)
print(airlines.columns.tolist())
airlines.head()

(6342, 6)
['IATA', 'ICAO', 'Airline', 'Call sign', 'Country', 'Comments']


,IATA,ICAO,Airline,Call sign,Country,Comments
0,DM,DWI,Arajet,Arajet,Dominican Republic,NaN
1,NaN,GBT,A-Jet Aviation Aircraft Management,GLOBETROTTER,Austria,NaN
2,NaN,AJR,A-Jet Aviation Company,JET MONGOLIA,Mongolia,NaN
3,NaN,SFM,A-Safar Air Services,AIR SAFAR,Nigeria,NaN
4,NaN,AJJ,A2 Jet Leasing,ATLANTIC JET,United States,NaN


In [26]:
airlines = airlines.copy()

airlines.columns = (
    airlines.columns.str.replace("\xa0", " ")
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

for col in airlines.columns:
    airlines[col] = (
        airlines[col]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.replace("\xa0", " ")
        .str.replace("*", "", regex=False)
        .str.strip()
    )
    airlines[col] = airlines[col].mask(airlines[col].isin(["nan", "None", ""]))

airlines = airlines.drop_duplicates()

print(airlines.columns.tolist())
print(airlines.shape)
print("Med IATA-kod:", airlines["iata"].notna().sum())
print("Med ICAO-kod:", airlines["icao"].notna().sum())
print("Upprepade IATA-koder:", airlines["iata"].dropna().duplicated().sum())
print("Upprepade ICAO-koder:", airlines["icao"].dropna().duplicated().sum())


['iata', 'icao', 'airline', 'call_sign', 'country', 'comments']
(6321, 6)
Med IATA-kod: 1542
Med ICAO-kod: 6219
Upprepade IATA-koder: 536
Upprepade ICAO-koder: 336


In [27]:
print([c for c in df.columns if "airline" in c])

used = df[["airline_operator__iata", "airline_operator__icao", "airline_operator__name"]].drop_duplicates()

check = used.merge(
    airlines[["icao", "airline", "country"]].drop_duplicates(subset="icao"),
    left_on="airline_operator__icao",
    right_on="icao",
    how="left",
)

print("Flygbolag i dina flyg:", len(used))
print("Hittade i Wikipedia (via ICAO):", check["airline"].notna().sum())
check[check["airline"].isna()]


['airline_operator__iata', 'airline_operator__icao', 'airline_operator__name']
Flygbolag i dina flyg: 59
Hittade i Wikipedia (via ICAO): 55


,airline_operator__iata,airline_operator__icao,airline_operator__name,icao,airline,country
19,OJ,NYX,Nyxair,NaN,NaN,NaN
55,FRO,FRO,FROSTAIR AIR APS,NaN,NaN,NaN
56,FT,FT,FROST Air,NaN,NaN,NaN
58,IV,GPX,GP Aviation,NaN,NaN,NaN
